In [25]:
# grouping by source

#!/usr/bin/env python
"""
Interdependence-strength summary figure: RF-SHAP and Granger causality (GC)
across period x scenario x region, from a 5-model CMIP6 ensemble.

Classification (per model, per directed link source -> target):
    RF-SHAP  Dv = shap_dependency, sign = sign(shap_direction)
             Strong  Dv > 0.03 | Moderate Dv > 0.02 | Weak Dv > 0.015
             -> +/-3, +/-2, +/-1, 0 = not significant
    GC       p = granger_p_value (positive only)
             Strong  p < 0.001 | Moderate p < 0.01  | Weak p < 0.05
             -> +3, +2, +1, 0 = not significant

Ensemble cell value = median class across models (an integer for an odd number
of models; for an even number it is rounded toward zero). A dot marks robust
links: at least AGREE_MIN models share the ensemble sign at >= weak strength.

Expected file names (CSV or Excel), e.g.
    djf_shap_granger_comparison_2021-2050_ESM2_TROP585.csv
    {season}_shap_granger_comparison_{period}_{model}_{region}{scenario}.csv/.xlsx

Usage
    python gc_shap_summary.py --indir /path/to/files --season djf --out fig_djf
    python gc_shap_summary.py --demo /mnt/user-data/uploads/<one file>.csv
Outputs: <out>.pdf, <out>.png (600 dpi), <out>_classes.csv (per model),
         <out>_ensemble.csv (per cell).
"""
import argparse
import glob
import os
import re

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Rectangle

# ============================================================ configuration
SHAP_THRESH = [(0.03, 3), (0.02, 2), (0.015, 1)]   # Dv > thr -> class
GC_THRESH = [(0.001, 3), (0.01, 2), (0.05, 1)]     # p  < thr -> class
AGREE_MIN = 5                                       # models (of 5) for the dot
DOT_MS = 2.5           # marker size (points) for robust-link dots
DOT_KW = dict(marker="o", ms=DOT_MS, mfc="k", mec="yellow", mew=0.3)

FNAME_RE = re.compile(
    r"^(?P<season>[A-Za-z]+)_shap_granger_comparison_(?P<period>\d{4}-\d{4})_"
    r"(?P<model>[^_]+)_(?P<region>[A-Za-z]+?)(?P<scenario>126|245|370|585)"
    r"\.(?:csv|xlsx|xls)$")


# region code in the file name -> column label (order = column order)
#REGIONS = {"TROP": "TROP", "SMZ": "SMZ", "NMZ": "NMZ",
#           "SSZ": "SSZ", "NSZ": "NSZ"}

REGIONS = {"TROP": "Tropics",
           "NSZ": "Northern Subtropics",
           "SSZ": "Southern Subtropics",
           "NMZ": "Northern Midlatitudes",
           "SMZ": "Southern Midlatitudes"}

COMBOS = [  # (scenario code, period, header)
    ("126", "2021-2050", "SSP1-2.6\n2021–2050"),
    ("126", "2071-2100", "SSP1-2.6\n2071–2100"),
    ("585", "2021-2050", "SSP5-8.5\n2021–2050"),
    ("585", "2071-2100", "SSP5-8.5\n2071–2100"),
]
SCEN_COLORS = {"126": "#1d3354", "585": "#980002"}

VARS = {  # file name -> label (order = row grouping)
    "temperature": "T", "precipitable_water": "Prw", "precipitation": "Pr",
    "evaporation": "ET", "soil_moisture": "SM", "runoff": "q",
}
GROUP_BY = "source"          # rows grouped by "target" or "source" variable

CLASS_LABELS = {-3: "Negative strong", -2: "Negative moderate", -1: "Negative weak",
                0: "Not significant",
                1: "Positive weak", 2: "Positive moderate", 3: "Positive strong"}
CLASS_COLORS = {-3: "#08306b", -2: "#4a89c3", -1: "#b5d4e9", 0: "#f2f2f2",
                1: "#f7c1a6", 2: "#dd6b56", 3: "#8c0d25"}
MISSING_COLOR = "white"

MM = 1 / 25.4
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans", "Arial", "DejaVu Sans"],
    "font.size": 7, "axes.titlesize": 7, "axes.labelsize": 7,
    "xtick.labelsize": 6, "ytick.labelsize": 6, "legend.fontsize": 6,
    "axes.linewidth": 0.5, "pdf.fonttype": 42, "ps.fonttype": 42,
})


# ============================================================ classification
def classify_shap(dv, direction):
    dv = np.asarray(dv, float)
    sign = np.sign(np.asarray(direction, float))
    cls = np.zeros(dv.shape, int)
    for thr, c in reversed(SHAP_THRESH):            # weak -> strong overwrite
        cls[dv > thr] = c
    return (cls * sign).astype(int)


def classify_gc(p):
    p = np.asarray(p, float)
    cls = np.zeros(p.shape, int)
    for thr, c in reversed(GC_THRESH):
        cls[p < thr] = c
    return cls


# ============================================================ input
def read_one(path):
    return pd.read_excel(path) if path.lower().endswith((".xlsx", ".xls")) \
        else pd.read_csv(path)


def load_dir(indir, season=None):
    rows = []
    files = sorted(glob.glob(os.path.join(indir, "**", "*"), recursive=True))
    for f in files:
        m = FNAME_RE.match(os.path.basename(f))
        if not m:
            continue
        meta = m.groupdict()
        if season and meta["season"].lower() != season.lower():
            continue
        d = read_one(f)
        d = d[d.target_variable != d.source_variable].copy()   # drop self-links
        for k, v in meta.items():
            d[k] = v
        rows.append(d)
    if not rows:
        #raise SystemExit(f"No files matching the naming pattern in {indir}")
        raise FileNotFoundError(f"No {season} files matching the naming pattern in {indir}")
    df = pd.concat(rows, ignore_index=True)
    df["shap_class"] = classify_shap(df.shap_dependency, df.shap_direction)
    df["gc_class"] = classify_gc(df.granger_p_value)
    df["link"] = df.source_variable + "->" + df.target_variable
    print(f"loaded {len(rows)} files | models: {sorted(df.model.unique())} | "
          f"regions: {sorted(df.region.unique())}")
    return df


def ensemble(df):
    """Per (scenario, period, region, link): median class + model agreement."""
    out = []
    for key, g in df.groupby(["scenario", "period", "region", "link"]):
        rec = dict(zip(["scenario", "period", "region", "link"], key), n_models=len(g))
        for meth in ("shap", "gc"):
            c = g[f"{meth}_class"].to_numpy()
            med = np.median(c)
            med = int(np.trunc(med))                 # toward zero if even n
            sgn = np.sign(med)
            agree = int(((np.sign(c) == sgn) & (c != 0)).sum()) if sgn != 0 else 0
            rec[f"{meth}_ens"] = med
            rec[f"{meth}_agree"] = agree
            rec[f"{meth}_robust"] = agree >= min(AGREE_MIN, len(g))
        out.append(rec)
    return pd.DataFrame(out)


# ============================================================ plotting
def ordered_links():
    v = list(VARS)
    if GROUP_BY == "target":
        return [(s, t) for t in v for s in v if s != t]
    return [(s, t) for s in v for t in v if s != t]


def draw_panel(ax, ens, meth, title):
    links = ordered_links()
    idx = ens.set_index(["scenario", "period", "region", "link"])
    nreg, gap = len(REGIONS), 0.6
    for g, (scn, per, lab) in enumerate(COMBOS):
        x0 = g * (nreg + gap)

        for j, reg in enumerate(REGIONS):
            for i, (s, t) in enumerate(links):
                key = (scn, per, reg, f"{s}->{t}")
                if key in idx.index:
                    r = idx.loc[key]
                    fc = CLASS_COLORS[int(r[f"{meth}_ens"])]
                    rob = bool(r[f"{meth}_robust"])
                else:
                    fc, rob = MISSING_COLOR, False
                ax.add_patch(Rectangle((x0 + j, i), 1, 1, fc=fc, ec="white", lw=0.35))
                if rob:
                    #ax.plot(x0 + j + .5, i + .5, "o", ms=DOT_MS, mfc="k", mec="white", mew=0.3)
                    #ax.plot(x0 + j + .5, i + .5, ls="", **DOT_KW)
                    ax.plot(x0 + j + .5, i + .5, "o", ms=DOT_MS, mfc="k", mec="none")
        ax.text(x0 + nreg / 2, -0.5, lab, ha="center", va="bottom", fontsize=6,
                linespacing=1.1, color=SCEN_COLORS.get(scn, "k"), fontweight="bold")
        
        for j, rl in enumerate(REGIONS.values()):
            ax.text(x0 + j + .5, len(links) + .3, rl.replace(" ", "\n"),
                    ha="center", va="top", rotation=90, linespacing=1.0,
                    fontsize=5.5, color="0.3")

        #for j, rl in enumerate(REGIONS.values()):
        #    ax.text(x0 + j + .5, len(links) + .3, rl, ha="center", va="top",
        #            fontsize=5.5, color="0.3")
    xmax = len(COMBOS) * (nreg + gap) - gap
    key_i = 1 if GROUP_BY == "target" else 0
    for i in range(1, len(links)):
        if links[i][key_i] != links[i - 1][key_i]:
            ax.plot([0, xmax], [i, i], color="k", lw=0.5)
    ax.set_xlim(0, xmax)
    ax.set_ylim(len(links), 0)
    ax.set_yticks(np.arange(len(links)) + .5,
                  [f"{VARS[s]} → {VARS[t]}" for s, t in links], fontsize=5)
    ax.set_xticks([])
    ax.tick_params(axis="y", length=0, pad=1.5)
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.annotate(title, (0, 1), xycoords="axes fraction", xytext=(0, 24),
                textcoords="offset points", fontsize=7, fontweight="bold",
                ha="left", va="bottom")

def make_figure(ens, out, methods=("gc", "shap"), season=None):
    titles = {"shap": "RF-SHAP dependency", "gc": "Granger causality"}
    n = len(methods)
    fig, axes = plt.subplots(n, 1, figsize=(183 * MM, 240 * MM), squeeze=False)
    fig.subplots_adjust(left=0.075, right=0.99, top=0.945, bottom=0.135, hspace=0.28)
    for ax, meth, letter in zip(axes[:, 0], methods, "abcd"):
        draw_panel(ax, ens, meth, titles[meth])
        ax.annotate(letter, (-0.07, 1), xycoords="axes fraction", xytext=(0, 23),
                    textcoords="offset points", fontsize=8, fontweight="bold",
                    ha="left", va="bottom")

    patch = lambda k, lab: Patch(fc=CLASS_COLORS[k], ec="0.6", lw=0.3, label=lab)
    #dot = plt.Line2D([], [], ls="", label="all models agree",
    #                 **{**DOT_KW, "ms": 4, "mec": "none"}) # no edge
    #dot = plt.Line2D([], [], ls="", label="all models agree",
    #                 **{**DOT_KW, "ms": 4, "mew": 0.5}) 
    #dot = plt.Line2D([], [], ls="", label="all models agree", **DOT_KW, "mew": 0.5)
    dot = plt.Line2D([], [], ls="", marker="o", ms=DOT_MS, mfc="k", mec="none",
                     label=f"all models agree")
    kw = dict(frameon=False, handlelength=1.2, handleheight=1.0, labelspacing=0.4,
              columnspacing=1.2, title_fontsize=6.5, alignment="left", markerscale=1.2)

    # ---- Granger causality legend (positive only)
    gc_h = [patch(3, "Strong"), patch(2, "Moderate"),
            patch(1, "Weak"), patch(0, "Not Significant"), dot]
    fig.legend(handles=gc_h, loc="lower left", bbox_to_anchor=(0.075, 0.01),
               ncol=2, title="Granger causality (GC)", **kw)

    # ---- RF-SHAP legend (positive | negative)
    sh_h = [patch(3, "Strong Positive"), patch(2, "Moderate Positive"),
            patch(1, "Weak Positive"),
            patch(-3, "Strong Negative"), patch(-2, "Moderate Negative"), patch(-1, "Weak Negative"),
            patch(0, "Not Significant"), dot]
    fig.legend(handles=sh_h, loc="lower left", bbox_to_anchor=(0.47, 0.01),
               ncol=3, title="RF-SHAP dependency", **kw)

    if season:
        fig.text(0.99, 0.985, season.upper(), fontsize=7, fontweight="bold",
                 ha="right", va="top")
    for ext in ("pdf", "png"):
        fig.savefig(f"{out}.{ext}", dpi=600)
    plt.close(fig)
    print("saved", out)

# ============================================================ main
DEFAULT_INDIR = "/Users/dianindrawati/Documents/Python/CMIP6_and_ERA5/Network Change_rev1"


def run(indir=DEFAULT_INDIR, season="jja", method="both", out=None):
    """One season -> one figure. Returns (per-model classes, ensemble table)."""
    out = out or f"fig_gc_shap_{season}"
    df = load_dir(indir, season)
    n_mod = df.groupby(["scenario", "period", "region", "link"]).size()
    if n_mod.max() > 5:
        print(f"WARNING: up to {n_mod.max()} rows per cell - duplicate files?")
    ens = ensemble(df)
    df.to_csv(f"{out}_classes.csv", index=False)
    ens.to_csv(f"{out}_ensemble.csv", index=False)
    methods = ("gc", "shap") if method == "both" else (method,)
    make_figure(ens, out, methods, season=season)
    return df, ens


def run_seasons(seasons=("jja", "djf"), indir=DEFAULT_INDIR, method="both",
                out_prefix="fig_gc_shap_rev1"):
    """Separate figure per season, in the given order (JJA first)."""
    results = {}
    for s in seasons:
        print(f"\n=== {s.upper()} ===")
        results[s] = run(indir=indir, season=s, method=method,
                         out=f"{out_prefix}_{s}")
    return results


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--indir", default=DEFAULT_INDIR)
    ap.add_argument("--seasons", nargs="+", default=["jja", "djf"])
    ap.add_argument("--method", default="both", choices=["both", "shap", "gc"])
    ap.add_argument("--out_prefix", default="fig_gc_shap")
    a, _ = ap.parse_known_args()
    return run_seasons(a.seasons, a.indir, a.method, a.out_prefix)


if __name__ == "__main__" and "ipykernel" not in __import__("sys").modules:
    main()

res = run_seasons()          # JJA figure first, then DJF




=== JJA ===
loaded 100 files | models: ['ESM2', 'LM', 'MIROC6', 'MM', 'SR5'] | regions: ['NMZ', 'NSZ', 'SMZ', 'SSZ', 'TROP']
saved fig_gc_shap_rev1_jja

=== DJF ===
loaded 100 files | models: ['ESM2', 'LM', 'MIROC6', 'MM', 'SR5'] | regions: ['NMZ', 'NSZ', 'SMZ', 'SSZ', 'TROP']
saved fig_gc_shap_rev1_djf


In [23]:
#!/usr/bin/env python
"""
Interdependence-strength summary figure: RF-SHAP and Granger causality (GC)
across period x scenario x region, from a 5-model CMIP6 ensemble.

Classification (per model, per directed link source -> target):
    RF-SHAP  Dv = shap_dependency, sign = sign(shap_direction)
             Strong  Dv > 0.03 | Moderate Dv > 0.02 | Weak Dv > 0.015
             -> +/-3, +/-2, +/-1, 0 = not significant
    GC       p = granger_p_value (positive only)
             Strong  p < 0.001 | Moderate p < 0.01  | Weak p < 0.05
             -> +3, +2, +1, 0 = not significant

Ensemble cell value = median class across models (an integer for an odd number
of models; for an even number it is rounded toward zero). A dot marks robust
links: at least AGREE_MIN models share the ensemble sign at >= weak strength.

Expected file names (CSV or Excel), e.g.
    djf_shap_granger_comparison_2021-2050_ESM2_TROP585.csv
    {season}_shap_granger_comparison_{period}_{model}_{region}{scenario}.csv/.xlsx

Usage
    python gc_shap_summary.py --indir /path/to/files --season djf --out fig_djf
    python gc_shap_summary.py --demo /mnt/user-data/uploads/<one file>.csv
Outputs: <out>.pdf, <out>.png (600 dpi), <out>_classes.csv (per model),
         <out>_ensemble.csv (per cell).
"""
import argparse
import glob
import os
import re

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Rectangle

# ============================================================ configuration
SHAP_THRESH = [(0.03, 3), (0.02, 2), (0.015, 1)]   # Dv > thr -> class
GC_THRESH = [(0.001, 3), (0.01, 2), (0.05, 1)]     # p  < thr -> class
AGREE_MIN = 5                                       # models (of 5) for the dot

FNAME_RE = re.compile(
    r"^(?P<season>[A-Za-z]+)_shap_granger_comparison_(?P<period>\d{4}-\d{4})_"
    r"(?P<model>[^_]+)_(?P<region>[A-Za-z]+?)(?P<scenario>126|245|370|585)"
    r"\.(?:csv|xlsx|xls)$")


# region code in the file name -> column label (order = column order)
#REGIONS = {"TROP": "TROP", "SMZ": "SMZ", "NMZ": "NMZ",
#           "SSZ": "SSZ", "NSZ": "NSZ"}

REGIONS = {"TROP": "Tropics",
           "NSZ": "Northern Subtropics",
           "SSZ": "Southern Subtropics",
           "NMZ": "Northern Midlatitudes",
           "SMZ": "Southern Midlatitudes"}

COMBOS = [  # (scenario code, period, header)
    ("126", "2021-2050", "SSP1-2.6\n2021–2050"),
    ("126", "2071-2100", "SSP1-2.6\n2071–2100"),
    ("585", "2021-2050", "SSP5-8.5\n2021–2050"),
    ("585", "2071-2100", "SSP5-8.5\n2071–2100"),
]
SCEN_COLORS = {"126": "#1d3354", "585": "#980002"}

VARS = {  # file name -> label (order = row grouping)
    "temperature": "T", "precipitable_water": "Prw", "precipitation": "Pr",
    "evaporation": "ET", "soil_moisture": "SM", "runoff": "q",
}
GROUP_BY = "target"          # rows grouped by "target" or "source" variable

CLASS_LABELS = {-3: "Negative strong", -2: "Negative moderate", -1: "Negative weak",
                0: "Not significant",
                1: "Positive weak", 2: "Positive moderate", 3: "Positive strong"}
CLASS_COLORS = {-3: "#08306b", -2: "#4a89c3", -1: "#b5d4e9", 0: "#f2f2f2",
                1: "#f7c1a6", 2: "#dd6b56", 3: "#8c0d25"}
MISSING_COLOR = "white"

MM = 1 / 25.4
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans", "Arial", "DejaVu Sans"],
    "font.size": 7, "axes.titlesize": 7, "axes.labelsize": 7,
    "xtick.labelsize": 6, "ytick.labelsize": 6, "legend.fontsize": 6,
    "axes.linewidth": 0.5, "pdf.fonttype": 42, "ps.fonttype": 42,
})


# ============================================================ classification
def classify_shap(dv, direction):
    dv = np.asarray(dv, float)
    sign = np.sign(np.asarray(direction, float))
    cls = np.zeros(dv.shape, int)
    for thr, c in reversed(SHAP_THRESH):            # weak -> strong overwrite
        cls[dv > thr] = c
    return (cls * sign).astype(int)


def classify_gc(p):
    p = np.asarray(p, float)
    cls = np.zeros(p.shape, int)
    for thr, c in reversed(GC_THRESH):
        cls[p < thr] = c
    return cls


# ============================================================ input
def read_one(path):
    return pd.read_excel(path) if path.lower().endswith((".xlsx", ".xls")) \
        else pd.read_csv(path)


def load_dir(indir, season=None):
    rows = []
    files = sorted(glob.glob(os.path.join(indir, "**", "*"), recursive=True))
    for f in files:
        m = FNAME_RE.match(os.path.basename(f))
        if not m:
            continue
        meta = m.groupdict()
        if season and meta["season"].lower() != season.lower():
            continue
        d = read_one(f)
        d = d[d.target_variable != d.source_variable].copy()   # drop self-links
        for k, v in meta.items():
            d[k] = v
        rows.append(d)
    if not rows:
        #raise SystemExit(f"No files matching the naming pattern in {indir}")
        raise FileNotFoundError(f"No {season} files matching the naming pattern in {indir}")
    df = pd.concat(rows, ignore_index=True)
    df["shap_class"] = classify_shap(df.shap_dependency, df.shap_direction)
    df["gc_class"] = classify_gc(df.granger_p_value)
    df["link"] = df.source_variable + "->" + df.target_variable
    print(f"loaded {len(rows)} files | models: {sorted(df.model.unique())} | "
          f"regions: {sorted(df.region.unique())}")
    return df


def ensemble(df):
    """Per (scenario, period, region, link): median class + model agreement."""
    out = []
    for key, g in df.groupby(["scenario", "period", "region", "link"]):
        rec = dict(zip(["scenario", "period", "region", "link"], key), n_models=len(g))
        for meth in ("shap", "gc"):
            c = g[f"{meth}_class"].to_numpy()
            med = np.median(c)
            med = int(np.trunc(med))                 # toward zero if even n
            sgn = np.sign(med)
            agree = int(((np.sign(c) == sgn) & (c != 0)).sum()) if sgn != 0 else 0
            rec[f"{meth}_ens"] = med
            rec[f"{meth}_agree"] = agree
            rec[f"{meth}_robust"] = agree >= min(AGREE_MIN, len(g))
        out.append(rec)
    return pd.DataFrame(out)


# ============================================================ plotting
def ordered_links():
    v = list(VARS)
    if GROUP_BY == "target":
        return [(s, t) for t in v for s in v if s != t]
    return [(s, t) for s in v for t in v if s != t]


def draw_panel(ax, ens, meth, title):
    links = ordered_links()
    idx = ens.set_index(["scenario", "period", "region", "link"])
    nreg, gap = len(REGIONS), 0.6
    for g, (scn, per, lab) in enumerate(COMBOS):
        x0 = g * (nreg + gap)

        for j, reg in enumerate(REGIONS):
            for i, (s, t) in enumerate(links):
                key = (scn, per, reg, f"{s}->{t}")
                if key in idx.index:
                    r = idx.loc[key]
                    fc = CLASS_COLORS[int(r[f"{meth}_ens"])]
                    rob = bool(r[f"{meth}_robust"])
                else:
                    fc, rob = MISSING_COLOR, False
                ax.add_patch(Rectangle((x0 + j, i), 1, 1, fc=fc, ec="white", lw=0.35))
                if rob:
                    #ax.plot(x0 + j + .5, i + .5, "o", ms=DOT_MS, mfc="k", mec="white", mew=0.3)
                    #ax.plot(x0 + j + .5, i + .5, ls="", **DOT_KW)
                    ax.plot(x0 + j + .5, i + .5, "o", ms=DOT_MS, mfc="k", mec="none")
        ax.text(x0 + nreg / 2, -0.5, lab, ha="center", va="bottom", fontsize=6,
                linespacing=1.1, color=SCEN_COLORS.get(scn, "k"), fontweight="bold")
        
        for j, rl in enumerate(REGIONS.values()):
            ax.text(x0 + j + .5, len(links) + .3, rl.replace(" ", "\n"),
                    ha="center", va="top", rotation=90, linespacing=1.0,
                    fontsize=5.5, color="0.3")

        #for j, rl in enumerate(REGIONS.values()):
        #    ax.text(x0 + j + .5, len(links) + .3, rl, ha="center", va="top",
        #            fontsize=5.5, color="0.3")
    xmax = len(COMBOS) * (nreg + gap) - gap
    key_i = 1 if GROUP_BY == "target" else 0
    for i in range(1, len(links)):
        if links[i][key_i] != links[i - 1][key_i]:
            ax.plot([0, xmax], [i, i], color="k", lw=0.5)
    ax.set_xlim(0, xmax)
    ax.set_ylim(len(links), 0)
    ax.set_yticks(np.arange(len(links)) + .5,
                  [f"{VARS[s]} → {VARS[t]}" for s, t in links], fontsize=5)
    ax.set_xticks([])
    ax.tick_params(axis="y", length=0, pad=1.5)
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.annotate(title, (0, 1), xycoords="axes fraction", xytext=(0, 24),
                textcoords="offset points", fontsize=7, fontweight="bold",
                ha="left", va="bottom")

def make_figure(ens, out, methods=("gc", "shap"), season=None):
    titles = {"shap": "RF-SHAP dependency", "gc": "Granger causality"}
    n = len(methods)
    fig, axes = plt.subplots(n, 1, figsize=(183 * MM, 240 * MM), squeeze=False)
    fig.subplots_adjust(left=0.075, right=0.99, top=0.945, bottom=0.135, hspace=0.28)
    for ax, meth, letter in zip(axes[:, 0], methods, "abcd"):
        draw_panel(ax, ens, meth, titles[meth])
        ax.annotate(letter, (-0.07, 1), xycoords="axes fraction", xytext=(0, 23),
                    textcoords="offset points", fontsize=8, fontweight="bold",
                    ha="left", va="bottom")

    patch = lambda k, lab: Patch(fc=CLASS_COLORS[k], ec="0.6", lw=0.3, label=lab)
    #dot = plt.Line2D([], [], ls="", label="all models agree",
    #                 **{**DOT_KW, "ms": 4, "mec": "none"}) # no edge
    #dot = plt.Line2D([], [], ls="", label="all models agree",
    #                 **{**DOT_KW, "ms": 4, "mew": 0.5}) 
    #dot = plt.Line2D([], [], ls="", label="all models agree", **DOT_KW)
    dot = plt.Line2D([], [], ls="", marker="o", ms=DOT_MS, mfc="k", mec="none",
                     label="all models agree")
    kw = dict(frameon=False, handlelength=1.2, handleheight=1.0, labelspacing=0.4,
              columnspacing=1.2, title_fontsize=6.5, alignment="left", markerscale=1.2)

    # ---- Granger causality legend (positive only)
    gc_h = [patch(3, "Strong"), patch(2, "Moderate"),
            patch(1, "Weak"), patch(0, "Not Significant"), dot]
    fig.legend(handles=gc_h, loc="lower left", bbox_to_anchor=(0.075, 0.01),
               ncol=2, title="Granger causality (GC)", **kw)

    # ---- RF-SHAP legend (positive | negative)
    sh_h = [patch(3, "Strong Positive"), patch(2, "Moderate Positive"),
            patch(1, "Weak Positive"),
            patch(-3, "Strong Negative"), patch(-2, "Moderate Negative"), patch(-1, "Weak Negative"),
            patch(0, "Not Significant"), dot]
    fig.legend(handles=sh_h, loc="lower left", bbox_to_anchor=(0.47, 0.01),
               ncol=3, title="RF-SHAP dependency", **kw)

    if season:
        fig.text(0.99, 0.985, season.upper(), fontsize=7, fontweight="bold",
                 ha="right", va="top")
    for ext in ("pdf", "png"):
        fig.savefig(f"{out}.{ext}", dpi=600)
    plt.close(fig)
    print("saved", out)

# ============================================================ main
DEFAULT_INDIR = "/Users/dianindrawati/Documents/Python/CMIP6_and_ERA5/Network Change_rev1"


def run(indir=DEFAULT_INDIR, season="jja", method="both", out=None):
    """One season -> one figure. Returns (per-model classes, ensemble table)."""
    out = out or f"fig_gc_shap_{season}"
    df = load_dir(indir, season)
    n_mod = df.groupby(["scenario", "period", "region", "link"]).size()
    if n_mod.max() > 5:
        print(f"WARNING: up to {n_mod.max()} rows per cell - duplicate files?")
    ens = ensemble(df)
    df.to_csv(f"{out}_classes.csv", index=False)
    ens.to_csv(f"{out}_ensemble.csv", index=False)
    methods = ("gc", "shap") if method == "both" else (method,)
    make_figure(ens, out, methods, season=season)
    return df, ens


def run_seasons(seasons=("jja", "djf"), indir=DEFAULT_INDIR, method="both",
                out_prefix="fig_gc_shap"):
    """Separate figure per season, in the given order (JJA first)."""
    results = {}
    for s in seasons:
        print(f"\n=== {s.upper()} ===")
        results[s] = run(indir=indir, season=s, method=method,
                         out=f"{out_prefix}_{s}")
    return results


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--indir", default=DEFAULT_INDIR)
    ap.add_argument("--seasons", nargs="+", default=["jja", "djf"])
    ap.add_argument("--method", default="both", choices=["both", "shap", "gc"])
    ap.add_argument("--out_prefix", default="fig_gc_shap")
    a, _ = ap.parse_known_args()
    return run_seasons(a.seasons, a.indir, a.method, a.out_prefix)


if __name__ == "__main__" and "ipykernel" not in __import__("sys").modules:
    main()

res = run_seasons()          # JJA figure first, then DJF




=== JJA ===
loaded 100 files | models: ['ESM2', 'LM', 'MIROC6', 'MM', 'SR5'] | regions: ['NMZ', 'NSZ', 'SMZ', 'SSZ', 'TROP']
saved fig_gc_shap_jja

=== DJF ===
loaded 100 files | models: ['ESM2', 'LM', 'MIROC6', 'MM', 'SR5'] | regions: ['NMZ', 'NSZ', 'SMZ', 'SSZ', 'TROP']
saved fig_gc_shap_djf
